# Test all_splits Function

In [1]:
# %%
import pandas as pd
import numpy as np
from pathlib import Path

from FIT_python.config import RAW_DIR
from FIT_python.splits_wrapper import all_splits
from FIT_python.split_utils import (
    group_stratified_kfold,
    create_train_test_split_otter
)

pd.set_option('display.max_columns', None)

# %% [markdown]
## 1. Übersicht der Datensätze

# %%
raw_dir = Path(RAW_DIR)
print(f"Raw data directory: {raw_dir}")
# Liste alle CSV- und Parquet-Dateien
file_paths = [p for p in raw_dir.iterdir() if p.suffix.lower() in ['.csv', '.parquet']]
print("Verfügbare Datendateien:")
for p in file_paths:
    print(f" - {p.name}")
print()

# Hilfsfunktion zum Laden
# Name aus splits keys kann z.B. 'Amur_Tiger' sein: wir normalisieren beides für Abgleich.
def load_full_df(name: str) -> pd.DataFrame:
    normalized = name.replace('_', ' ').lower()
    for p in file_paths:
        if normalized in p.stem.lower():
            print(f"Lade kompletten DataFrame für {name} von {p.name}")
            if p.suffix.lower() == '.csv':
                return pd.read_csv(p)
            else:
                return pd.read_parquet(p)
    print(f"Kein komplettes DataFrame gefunden für {name}")
    return None

# %% [markdown]
## 2. Test: all_splits-Funktion

# %%
splits = all_splits(raw_dir)
print(f"all_splits liefert {len(splits)} Datensätze")
assert isinstance(splits, dict), "all_splits sollte ein dict zurückgeben"

for name, parts in splits.items():
    print(f"\n=== Datensatz: {name} ===")
    # prüfe keys
    print(f"Keys: {list(parts.keys())}")
    assert 'train' in parts and 'test' in parts, "Fehlende Keys 'train'/'test'"
    train_df = parts['train']
    test_df = parts['test']
    print(f"Train-Größe: {len(train_df)}, Test-Größe: {len(test_df)}")
    # Show sample rows
    display(train_df.head(2))
    display(test_df.head(2))
    # vollständiges DF prüfen
    df_all = load_full_df(name)
    if df_all is not None:
        total = len(df_all)
        sum_splits = len(train_df) + len(test_df)
        print(f"Total: {total}, sum(train+test): {sum_splits}")
        assert total == sum_splits, (
            f"Anzahl Zeilen stimmt nicht überein für {name}: {total} vs {sum_splits}"
        )
    # prüfe Gruppenüberschneidung
    train_ids = set(train_df['individual_id'])
    test_ids = set(test_df['individual_id'])
    overlap = train_ids & test_ids
    print(f"Anzahl überlappender individual_id: {len(overlap)}")
    assert not overlap, (
        f"Train und Test teilen sich individual_id(s) in {name}: {overlap}"
    )
    print(f"✔ Train/Test-Check erfolgreich für {name}")



Raw data directory: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\data\raw
Verfügbare Datendateien:
 - Amur Tiger.csv
 - Bengal tiger.csv
 - cheetah.csv
 - Eurasian Otter.csv
 - Giant Panda.csv
 - Jubatus.csv
 - Lowlandtapir.csv
 - Mountain Lion.csv
 - White Rhino.csv

all_splits liefert 9 Datensätze

=== Datensatz: Amur_Tiger ===
Keys: ['train', 'test']
Train-Größe: 476, Test-Größe: 129


,id,species,individual_id,trail,sex,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,v104,v105,v106,v107,v108,v109,v110,v111,v112,v113,v114,v115,v116,v117,v118,area1,area2,area3,area4,area5,area6,area7,area8,area9,area10
0,Amur_Tiger_1,Panthera tigris altaica,M541,M541 A,M,"3,205367883","2,887805367","2,708196628","2,910179401","2,082426065","2,014035107","1,850129064","1,867721812","2,813425504","8,333106222","2,844254261","2,48703483","2,611681899","6,71836502","8,269086532","9,354471226","6,678411451","5,328498719","2,940587813","5,473713448","3,880757779","5,277828525","1,802338609","1,242021228","3,497886532","4,158041247","3,79637718","4,061221397","3,476829496","2,760226505","2,160531154","3,571331238","4,244318199","3,531581185","3,929019308","6,373044791","6,776526774","3,466798191","4,225768997","6,074145532","5,42708089","6,134202866","4,999605653","4,887503522","2,764823498","7,112072774","9,870969415","8,963179756","9,066149465","8,364443617","7,410599662","2,115406246","1,796317319","1,85894625","1,876165839","1,704456054","1,851601511","1,320678759","2,117723936","1,721954428","1,598512384","1,466191363","1,778018194","1,771876489","1,631252128","1,382382285","2,129530512","6,428396228","7,158593603","4,291354682","3,627008697","3,887565597","3,394578613","1,000926942","1,237656083","2,629066848","5,585272629","4,92429362","4,523459609","4,028190309","68,59283706","78,43811786","47,64229965","63,76486329","89,84198537","81,5744696","86,08342968","95,77133489","29,99868154","23,87164956","22,2848597","101,4410439","101,6583821","25,29945544","19,26174016","21,96225092","7,065631361","4,985562315","5,579867757","2,179119952","2,740408425","4,52332475","7,527868926","7,759268699","2,741203899","5,976071384","8,742267436","10,51687642","8,386700075","5,474249301","8,262228487","12,05119368","10,50956625","8,551649918","8,618440857","11,27170501","12,99346728","10,94192514","88,96798317","3,324321673","2,791252875","2,500735831","2,664253867","34,32352588","61,55854491","17,6899262","21,60400292","47,01252548"
1,Amur_Tiger_2,Panthera tigris altaica,M541,M541 A,M,"3,165034302","3,16130854","3,192838944","3,173958289","2,18526282","2,378921088","2,044270323","2,332393693","1,893426989","7,870396945","3,372349793","2,532027442","3,107541507","6,179521779","7,816907315","9,726374786","6,94001886","4,950323005","2,86658431","5,144365864","4,582008922","4,938051004","1,693366848","1,452736973","4,490142358","4,222682441","3,691032552","4,886396214","4,20842461","1,592539056","2,023409876","3,481029936","4,439285414","3,626936628","4,820103175","7,351349028","6,642758715","3,984207662","4,397350622","6,896000001","5,380575329","5,771565464","4,979430219","6,366989756","3,688196379","7,299868359","10,54271425","9,818270465","8,272369281","7,998727094","8,418529285","1,940154942","1,964302168","1,724016823","2,060349101","1,817849659","2,265754723","1,780512793","2,028064074","1,920618585","1,660645324","1,777863219","2,269912074","2,107849667","1,888440997","1,710290356","2,184723449","5,255235361","7,11178431","4,218127858","3,819242035","4,842321113","3,33244556","0,553738112","1,284648126","3,427349577","4,906588918","6,023930949","3,643250731","5,051974346","73,30389028","75,29787692","42,64080507","64,05530465","83,65980825","77,43885833","79,31235135","98,08811205","32,52222555","20,87121498","28,27961721","98,10169592","111,4830186","22,75639157","13,61592666","25,25326723","6,845466805","6,117225656","4,903490407","2,912964598","3,107162238","5,728830376","7,233862085","9,370036124","1,553581119","5,291885686","9,749248587","11,38222254","8,523844118","5,222165553",

,id,species,individual_id,trail,sex,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,v104,v105,v106,v107,v108,v109,v110,v111,v112,v113,v114,v115,v116,v117,v118,area1,area2,area3,area4,area5,area6,area7,area8,area9,area10
0,Amur_Tiger_16,Panthera tigris altaica,F536,F536 A,F,"2,939688272","2,792609515","3,134023941","3,052597954","1,96289671","2,116356462","2,126788044","1,8115639","2,682115621","6,464217333","1,976273563","2,257083763","2,433674611","4,45920345","6,01581537","8,315961434","6,095694459","3,903391652","2,112423718","4,390414422","3,925547012","3,902368498","1,674663477","1,2764225","3,659985361","3,119193028","3,922570713","3,596560854","3,267375711","1,757228198","1,725149353","2,494830673","3,287633216","3,901017964","3,832884571","6,313948624","5,409813501","2,394096648","3,953700022","6,877091849","5,804584956","4,602499462","4,039247829","5,29643904","3,262576973","5,362855823","8,807705045","7,781748029","6,654289722","6,341595494","6,611708948","1,85922211","1,763812963","1,930414103","1,50596711","1,745192247","1,72086782","1,591701352","1,948887386","1,918366011","1,643816608","2,141602163","1,883997816","1,748209003","1,91561724","1,193665658","2,181097756","4,979724509","5,584652932","3,361669671","3,855755026","3,488165265","2,247147446","0,089367218","2,011780033","2,671062164","3,169845416","5,788413369","2,065999644","5,32568797","61,4221875","72,95927638","43,85942024","74,71839227","88,21008939","80,53767779","84,95754893","102,1549417","30,25190048","25,96048276","22,92205836","102,6724501","106,3174874","21,71112941","18,62554868","21,33200025","5,646956767","5,050123938","3,167805016","4,086009368","1,698678052","4,728752415","5,968328291","7,759370075","1,197192005","4,131919586","7,763307842","9,45850767","6,460955871","3,998408334","6,840622425","10,69708071","8,958258786","7,391687614","6,933050916","9,226126288","12,21462026","10,09868263","63,40211618","2,849675407","2,940308574","3,294410252","2,583045274","18,76949667","39,5013068","11,00666048","12,18699941","38,13232442"
1,Amur_Tiger_17,Panthera tigris altaica,F536,F536 A,F,"3,049648134","2,840854968","3,024883732","2,874196295","1,815693634","1,998549855","1,996495964","1,998549855","2,164971392","6,477484547","2,410462165","1,838150503","1,763991565","4,422133858","5,854392723","7,048338675","4,971748345","3,251135375","2,603257348","3,363348168","3,684990508","3,217454936","1,579417348","1,383360376","3,572602463","3,383135732","3,046058552","3,929700248","2,779373825","1,565754834","1,878138973","2,279837435","3,309436851","3,162940699","4,214815466","5,639834979","5,303419048","2,459885262","3,622166303","5,840852798","5,205922246","4,327066055","3,842164064","4,728309455","3,072298772","5,487187154","7,98621218","7,603389462","6,582270336","6,486697162","6,255809451","1,817951804","1,792446945","1,836661921","1,65189226","1,516054392","2,11037991","1,796003895","1,552010898","1,750762414","1,902495391","1,667547753","1,917769056","1,586865788","1,702982537","1,546714922","2,161599817","4,850900451","5,355594897","3,193222905","3,218700204","3,999516304","2,566742416","0,452740741","0,99418399","2,810364787","3,985764439","5,22376703","3,021116723","4,259774579","66,63115401","72,84771079","48,50686463","64,86198136","90,25351949","87,81687125","86,81988913","98,81298325","35,05847076","25,19768733","32,73415148","101,7542121","113,5157242","23,7739616","17,77419756","27,42502531","5,284043797","4,982098437","3,98567875","2,264590199","2,959064526","4,921709365","5,344432869","7,578828532","1,630504943","4,166845966","7,192703535","8,497978445","6,495826535","3,802593937","6,8

Lade kompletten DataFrame für Amur_Tiger von Amur Tiger.csv
Total: 605, sum(train+test): 605
Anzahl überlappender individual_id: 0
✔ Train/Test-Check erfolgreich für Amur_Tiger

=== Datensatz: Bengal_tiger ===
Keys: ['train', 'test']
Train-Größe: 401, Test-Größe: 185


,id,species,individual_id,trail,sex,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,v104,v105,v106,v107,v108,v109,v110,v111,v112,v113,v114,v115,v116,v117,v118,area1,area2,area3,area4,area5,area6,area7,area8,area9,area10
0,Bengal_tiger_1,Panthera tigris tigris,Alex,Alex a,F,"3,903835962","4,492925497","5,314363917","3,758673238","3,10804025","2,974448679","2,569874714","3,222925521","2,248832852","9,01308031","1,846835688","2,943724047","2,377193683","7,737856364","8,966543238","9,181956313","7,809491174","6,352584969","2,613958268","6,4321434","2,749812913","6,295252023","2,261540716","1,525422494","4,202493452","4,19162012","3,091542132","3,837776439","3,619075461","3,244889285","3,900195342","3,392356567","4,91799964","3,474281367","5,337169875","7,368569784","7,271927348","4,057871446","4,010383125","5,819036389","6,104267732","6,039599934","4,231532657","4,541375757","3,855434954","6,798204657","8,771533276","9,670025336","9,770791359","8,483298932","6,859675411","2,717902355","1,787023779","3,199923288","2,132098022","2,813692932","2,749342259","3,101863739","2,090720253","3,177648736","2,663256288","2,899966837","3,067786411","2,266764656","2,878825829","1,877729662","2,782214323","7,002935917","7,110734698","4,355797971","3,355311566","4,631894572","3,646520889","0,801112853","1,373000985","3,300783625","5,955215588","6,294436729","5,126777274","5,126777274","97,30575953","85,73542977","39,49823558","43,19600488","100,4814879","93,83996872","90,47350806","96,43771906","27,83896703","19,56923429","24,43456074","107,204569","104,1736818","26,93152464","18,39203064","23,18803167","8,744201623","5,69126916","5,955102829","1,95990726","4,334410287","5,766650208","8,668820574","8,142549061","4,111032488","7,500414323","9,036691284","9,744658903","8,923744022","6,84526747","9,121418352","14,43547078","12,24965232","10,25355455","10,69169301","13,3877782","14,78324031","12,18109696","118,0242634","5,540244907","6,533227264","6,810239509","5,770378157","43,56692332","67,40205131","15,93317612","24,05105549","66,73153962"
1,Bengal_tiger_2,Panthera tigris tigris,Alex,Alex a,F,"3,696379322","4,78859404","4,711450055","3,912352139","2,729365781","2,45918853","2,897963662","3,485160973","2,611288457","8,181156069","2,197497828","2,79289276","2,266225943","6,934776565","7,728086885","8,87233495","7,027548176","5,558131454","2,169955431","5,583452167","3,288882783","5,502269418","2,287794692","1,454745388","4,209369832","3,79459246","2,820723705","4,325286274","3,587377345","2,477314412","3,007867576","3,359412055","4,300422896","2,862707017","5,653518994","7,472633099","7,047528035","3,246983112","3,754748255","6,147801382","6,352987346","5,418841507","3,722138797","4,637145576","3,748691233","6,238271577","8,613221948","9,048751296","9,100322697","7,762469763","7,313662575","2,445868913","2,241016861","2,716848081","1,732362382","2,686356173","2,899678506","3,361102751","1,804641845","3,306142173","2,125547578","3,068175841","2,414602115","2,007329485","3,518015704","1,474126357","3,073032565","6,567130382","6,924631181","3,845435133","3,166673843","5,006319259","3,434880518","0,786038488","0,948667141","3,422998972","5,583364831","6,073795525","4,070469686","5,276534778","96,98848973","83,93037622","33,61116424","49,40034603","87,66269414","91,80399512","90,91665426","103,3669307","29,36255098","17,91250939","28,68545628","107,5821121","105,7902256","27,45794315","14,48044344","26,48059757","7,763476029","5,916823964","5,577642972","1,43209752","4,635473551","5,389209088","8,291090905","8,253404128","3,391809916","6,858993385","8,707195201","9,700081469","7,951999218","6,079146957

...

,id,species,individual_id,trail,sex,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,area1,area2,area3,area4,area5,area6,area7,area8,area9,area10,v77_v78,v79_v80,v73_v74_v75_v76,v22_v23_v24_v25,v94_v95,v96_v97_v98,v22_25_v77_78,v22_25_v10,v94_v95_v101_v102,v22_25_v10_v68,v2_v3,v1_v2_v3_v4,v1_v2_v3,v2_v3_v4,v93_v2,v93_ar_4,v93_v2_v3,ar10_ar8,v93_ar10_ar8,93_94_95
0,Mountain_Lion_20,Puma concolor,Female Archback 5,5A,F,"2,336960396","2,588656756","2,742678158","2,656920229","1,509070588","1,625960561","1,841418976","2,096394133","1,222355185","5,228066633","1,448534911","1,553235518","1,5687074","4,347975332","5,228867288","5,391805332","3,786818838","3,375608394","1,853258894","3,263052218","2,128753114","9,406280638","4,288833392","3,056547648","15,80433982","2,326603009","1,836515988","2,614033316","1,456294777","2,020204248","2,3579484","1,353613141","2,456536503","1,994376142","3,536262469","3,797179046","3,680126413","2,499084642","1,927055073","2,091427864","2,885179773","3,719376336","2,728200151","1,875679642","1,675696223","3,412342691","4,691319159","4,873654863","5,53357566","4,985940354","4,043356276","0,98843068","1,770815496","1,275217071","1,458062662","1,446755383","1,684326893","1,567680692","1,423192844","1,215221526","1,809667993","1,842992166","1,79483806","1,373685255","1,756207571","1,48692933","2,106507634","3,914597867","4,554019116","2,449685402","1,882561066","2,892490262","1,984712758","9,725331328","8,576410898","1,6884099","16,40291967","9,108190661","18,47711867","12,63885292","75,55908009","79,65146126","46,87604083","57,56487908","86,90735588","89,95861414","87,27581241","115,4546771","26,10122485","19,83818501","26,39532232","110,76166","117,5967761","4,035065146","4,161953987","3,578265318","0,456799828","3,248354332","3,755909696","4,441109438","5,151686948","2,13173253","4,771020424","38,431269","5,904704783","2,103083621","2,492400433","2,650188057","16,58573342","21,03502241","7,702821943","8,806481529","23,87552167","25,51111033","31,11597159","21,97486488","32,55600149","8,197019134","7,283419478","1,276149924","6,227158868","1,12543554","41,69866599","5,331334913","10,32521554","7,66829531","7,988255142","45,42772072","47,18213598","22,05766061","16,17269972","7,271313891","14,34628543"
1,Mountain_Lion_21,Puma concolor,Female Archback 5,5A,F,"2,519577262","2,488589626","2,672398212","2,596059786","1,161934196","1,701386984","1,677017041","1,7484836","1,617076451","5,96406971","1,603729715","1,709930281","1,704595738","4,494481538","5,243019472","5,535117232","4,494481538","3,258373683","1,984645789","2,999324608","2,535792623","6,977479332","4,241885345","1,923823881","12,16594579","2,178550846","3,024731092","3,607823921","0,833715759","2,291029469","1,968235921","1,761354008","2,620502045","2,556323635","3,999362426","3,391191848","4,26456574","2,825966675","1,763294553","2,594663072","4,287386226","4,430401095","2,943228985","1,754238578","2,765152797","3,589500983","5,604122864","6,515107845","6,384977816","5,721910084","3,907957414","1,533832183","1,268936706","1,376619761","1,365726719","1,088104249","1,72941971","1,557635872","1,648535255","1,587749091","1,639672098","1,678319916","1,40775888","1,546840013","1,680055511","1,101745593","1,888706731","4,320578117","5,306627125","2,165422615","2,855190077","3,619833308","2,032065838","7,656207513","5,890036755","2,949899489","14,26300069","5,8100235","15,48423068","8,115068331","79,79633991","97,8636542","46,72752228","53,47613781","94,91656601","77,47119229","90,57872557","98,97262661","24,33032955","32,45199558","40,52212043","101,8979093","129,7561867","4,507754905","3,832941296"

Lade kompletten DataFrame für Mountain_Lion von Mountain Lion.csv
Total: 535, sum(train+test): 535
Anzahl überlappender individual_id: 0
✔ Train/Test-Check erfolgreich für Mountain_Lion

=== Datensatz: White_Rhino ===
Keys: ['train', 'test']
Train-Größe: 914, Test-Größe: 359


,id,species,individual_id,trail,sex,var_01,var_02,var_03,var_04,var_05,var_06,var_07,var_08,var_09,var_10,var_11,var_12,var_13,var_14,var_15,var_16,var_17,var_18,var_19,var_20,var_21,var_22,var_23,var_24,var_25,var_26,var_27,var_28,var_29,var_30,var_31,var_32,var_33,v34,var_35,var_36,var_37,var_38,var_39,var_40,var_41,var_42,var_43,var_44,var_45,var_46,var_47,var_48a,var_49a,var_50a,var_51,var_52,var_53,var_54a,var_55a,var_56,var_57,var_58,var_59,var_60a,var_61a,var_62,var_63,var_64,var_65a,var_66a,var_67,var_68a,var_69,var_70,var_71,var_72a,var_73,var_74,var_75a,var_76,var_77,var_48ax49a,var_54ax55a,var_58x60,var_66ax67,var_68ax69,var_70x71x72a,var_73x74x75a,var_15x16,var_17x18,var_15x16x17x18,cv97,var_26x27,var_29x30,var_32x33_2,var_31x32x33_2,var_5x6_2,var_5x6x7_2,cv82_2,var_5x6x7x8x9_2,var_5x6x7x8x9x10_2,var_6x7x8_2,var_6x7x8x9_2,cv78_2,var_2x3,cv90_2,var_12x13_2,cv103_2,cv104_2,var_16x17_2,cv96_2,var_6x7x8x9x10_2,var_56x58x60_2,var_59x61a_2,var_9x10_2,cv98_2
0,White_Rhino_1,Ceratotherium simum,Bianca,NaN,F,"3,95","1,97","2,71","5,23","2,15","3,43","1,69","2,6","4,57","0,96","2,97","1,46","2,56","5,8","1,46","4,26","3,25","1,83","6,03","5,07","2,83","4,5","3,49","3,15","1,1","6,22","5,03","1,6","6,5","4,99","4,61","2,33","4,89","8,31","7,44","7,02","6,03","6,14","5,52","6,8","5,73","5,63","4,59","8,2","7,74","6,56","9,45","46,54","42,98","91,74","37,41","48,22","93,41","59,17","57,19","29,2","34,6","19,98","21,45","76,21","65,73","81,32","84,57","84,74","137,72","61,25","18,92","71,69","23,53","42,24",13,"44,3","34,72","10,63","39,1","83,35","87,58","89,52","116,36","96,19","80,17","95,22","99,54","84,45","5,72","5,08","10,8","7,99","11,25","11,49","7,22","11,83","5,58","7,27","9,87","14,44","15,4","7,72","12,29","13,86","4,68","12,79","4,02","11,66","12,53","7,51","11,1","13,25","125,39","87,18","5,53","13,95"
1,White_Rhino_2,Ceratotherium simum,Bianca,NaN,F,"4,19","2,41","2,95","4,97","2,91","4,03","2,25","2,71","4,38","1,01","3,91","2,05","2,6","5,7","2,52","4,5","3,64","1,71","7,17","5,39","3,57","5,7","3,84","3,19","1,98","7,29","5,39","1,63","7,45","5,4","4,81","2,87","5,43","10,03","8,76","8,28","6,7","7,1","6,28","7,52","6,39","6,06","5,4","8,91","7,87","6,72","10,06","45,62","37,95","91,77","41,68","48,25","88,7",60,"53,55","29,14",37,"20,43","22,83","72,65","67,17","85,71","87,94","88,84","137,04","64,25","19,45","69,78","23,52","37,11","13,74","45,56","31,19","11,91","43,6","86,87",90,"83,57","113,55","93,08","83,7","93,3","96,41","86,7","7,02","5,35","12,37","9,54","12,68","12,85","8,3","13,11","6,94","9,19","11,9","16,28","17,29","8,99","13,37","14,52","5,36","14,26","4,65","13,38","13,91","8,14","12,56","14,38","122,22",90,"5,39","16,29"


,id,species,individual_id,trail,sex,var_01,var_02,var_03,var_04,var_05,var_06,var_07,var_08,var_09,var_10,var_11,var_12,var_13,var_14,var_15,var_16,var_17,var_18,var_19,var_20,var_21,var_22,var_23,var_24,var_25,var_26,var_27,var_28,var_29,var_30,var_31,var_32,var_33,v34,var_35,var_36,var_37,var_38,var_39,var_40,var_41,var_42,var_43,var_44,var_45,var_46,var_47,var_48a,var_49a,var_50a,var_51,var_52,var_53,var_54a,var_55a,var_56,var_57,var_58,var_59,var_60a,var_61a,var_62,var_63,var_64,var_65a,var_66a,var_67,var_68a,var_69,var_70,var_71,var_72a,var_73,var_74,var_75a,var_76,var_77,var_48ax49a,var_54ax55a,var_58x60,var_66ax67,var_68ax69,var_70x71x72a,var_73x74x75a,var_15x16,var_17x18,var_15x16x17x18,cv97,var_26x27,var_29x30,var_32x33_2,var_31x32x33_2,var_5x6_2,var_5x6x7_2,cv82_2,var_5x6x7x8x9_2,var_5x6x7x8x9x10_2,var_6x7x8_2,var_6x7x8x9_2,cv78_2,var_2x3,cv90_2,var_12x13_2,cv103_2,cv104_2,var_16x17_2,cv96_2,var_6x7x8x9x10_2,var_56x58x60_2,var_59x61a_2,var_9x10_2,cv98_2
0,White_Rhino_11,Ceratotherium simum,Kaiser 03,KAI 2,M,"5,063695359","2,370136713","3,224851996","6,886431448","1,763436301","5,008139836","2,462675466","2,76252347","6,346540462","0,444790303","5,113605855","2,577523448","2,577301843","6,906366745","3,361731975","3,054111256","2,625972106","1,546640822","7,753491651","4,642880885","3,360247068","6,902084784","3,086905369","3,861982987","1,881360468","8,381473929","4,881133975","2,066740871","9,236296737","5,119698932","5,967082025","5,088950113","5,755433118","10,32514797","8,187103372","9,255798529","6,808475799","6,567855347","7,261738426","8,162122989","6,753930708","5,795595985","5,716848109","11,3347859","8,600632995","7,797619521","12,34455224","48,67003759","34,54730268","106,8789499","59,74800071","61,12991775","93,75626108","56,89538636","42,99339457","32,75660045","47,35461862","21,79900068","21,16903508","83,89170256","73,77547264","75,54544417","83,81934488","90,81009384","137,0319642","53,14026168","20,16781353","63,25649161","22,80022223","36,79775013","22,44076521","47,45340945","32,87390942","19,25674007","41,81263668","74,30929676","85,05549228","83,21734026","99,88878093","105,6907032","73,30807521","86,05671383","106,6919248","93,94328617","6,415843231","4,172612927","10,58845616","9,988990153","13,2626079","14,35599567","10,84438323","16,81146526","6,771576137","9,234251603","11,99677507","18,34331554","18,78810584","10,23333877","16,57987923","17,54511552","5,594988709","17,17479789","5,154825291","13,82959377","14,9160537","5,680083362","12,39637254","17,02466954","138,4473037","94,94450772","6,791330765","17,21070924"
1,White_Rhino_12,Ceratotherium simum,Kaiser 03,KAI 2,M,"5,878072706","2,389639721","2,761429285","7,729274747","2,607760108","5,464068308","2,233667123","2,688061744","6,03973475","1,265666636","5,347999275","2,213557319","2,736027074","7,298680101","0,276331392","5,210845003","2,613913072","3,317951422","7,301475595","5,647794221","3,802219019","6,426603084","3,685757578","3,430772165","2,184776527","8,039549518","5,530647645","1,585741735","8,8917068","5,38533078","6,888178942","4,390036911","4,998503315","10,9167913","9,82356913","9,145953272","7,83464486","6,426362126","7,160965475","8,023203869","7,344569935","7,607142984","6,021407239","11,49409996","9,086400487","8,585102032","12,22722187","41,97587801","35,0934974","107,1182627","45,90601995","65,13990113","87,82634027","52,96940626","46,12563025","36,68529196","44,21967153","21,3241743","18,55301548","87,00439747","68,06301568","71,42353218","93,63186489","90,38648546","140,1228102","53,11841275","18,75850973","72,05979454","21,11868005","39,46275434","19,05157845","49,60874473","36,78086767","16,73200663","33,30865111","71,67142823","93,38396884","77,06937541","99,09503651","108,3285718","71,87692248","93,17847458","108,1230775","86,82152542","5,487176395","5,931864494","11,41904089","10,11236066","13,57019716","14,27703758","9,388540226","16,27671917","8,071828416","10,30549554","12,99355728","19,03329203","20,29895867"

Lade kompletten DataFrame für White_Rhino von White Rhino.csv
Total: 1273, sum(train+test): 1273
Anzahl überlappender individual_id: 0
✔ Train/Test-Check erfolgreich für White_Rhino


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold

from FIT_python.config import RAW_DIR
from FIT_python.splits_wrapper import all_splits

NUM_FOLDS = 3
GLOBAL_RANDOM_SEED = 42
GROUP_COL = "individual_id"

def group_stratified_kfold(
    df: pd.DataFrame,
    n_splits: int = NUM_FOLDS,
    random_state: int = GLOBAL_RANDOM_SEED,
    group_col: str = GROUP_COL,
    stratify_col: str = "sex"
) -> pd.DataFrame:
    """
    Assigns a 'Fold' column via
      - StratifiedGroupKFold on groups, stratified by stratify_col,
        if every class has at least n_splits groups.
      - Otherwise GroupKFold on groups only.
    """
    # Build DataFrame of unique groups and their class
    individuals = (
        df[[group_col, stratify_col]]
        .drop_duplicates(subset=[group_col])
        .fillna({stratify_col: "unknown"})
    )
    class_counts = individuals[stratify_col].value_counts()
    use_strat = (class_counts >= n_splits).all()

    if use_strat:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_fn = lambda: splitter.split(individuals, individuals[stratify_col], groups=individuals[group_col])
    else:
        splitter = GroupKFold(n_splits=n_splits)
        split_fn = lambda: splitter.split(individuals, groups=individuals[group_col])

    fold_series = pd.Series(-1, index=df.index, name="Fold")
    for fold, (_, val_idx) in enumerate(split_fn()):
        fold_groups = individuals.iloc[val_idx][group_col]
        fold_series.loc[df[group_col].isin(fold_groups)] = fold

    return df.assign(Fold=fold_series)


# Generate splits and summarize by fold
raw_dir = Path(RAW_DIR)
splits = all_splits(raw_dir)

for name, parts in splits.items():
    print(f"=== Dataset: {name} ===")
    df_train = parts['train'].copy()

    # Ensure 'sex' column exists
    if 'sex' not in df_train.columns:
        df_train['sex'] = 'unknown'

    # Create debug stratify column if missing
    if 'proportion' not in df_train.columns:
        ids = pd.to_numeric(df_train['individual_id'], errors='coerce').fillna(0)
        df_train['proportion'] = np.floor(ids % NUM_FOLDS)

    folded = group_stratified_kfold(df_train, n_splits=NUM_FOLDS)

    # Summary per individual
    summary = (
        folded
        .groupby(['individual_id', 'sex', 'Fold'])
        .size()
        .reset_index(name='row_count')
    )
    print("Summary (individual_id, sex, Fold, row_count):")
    display(summary)

    # Unique individual counts per fold and sex
    counts = (
        folded
        .drop_duplicates(['individual_id', 'sex', 'Fold'])
        .groupby(['Fold', 'sex'])['individual_id']
        .nunique()
        .unstack(fill_value=0)
    )
    print("Unique individuals per Fold and sex:")
    display(counts)
    print()


=== Dataset: Amur_Tiger ===
Summary (individual_id, sex, Fold, row_count):


,individual_id,sex,Fold,row_count
0,F1020,F,0,23
1,F1022,F,1,16
2,F1152,F,2,11
3,F140,F,1,13
4,F200,F,0,8
5,F2013,F,2,19
6,F202,F,0,14
7,F214,F,2,14
8,F256,F,1,9
9,F342,F,0,11


Unique individuals per Fold and sex:


sex,F,M
Fold,,
0,6,6
1,6,6
2,6,5



=== Dataset: Bengal_tiger ===
Summary (individual_id, sex, Fold, row_count):


...

sex,F,M
Fold,,
0,5,4
1,5,5
2,4,5



=== Dataset: White_Rhino ===
Summary (individual_id, sex, Fold, row_count):


,individual_id,sex,Fold,row_count
0,Berta,F,0,32
1,Bianca,F,1,10
2,Brutus 03,M,2,11
3,Bull no 4 02,M,0,14
4,Felicia 02,F,1,13
5,Fiela,F,2,21
6,Francisca,F,0,21
7,Frauke 03,F,1,30
8,Gert,M,2,29
9,Gesa 02,F,1,19


Unique individuals per Fold and sex:


sex,F,M
Fold,,
0,5,6
1,8,2
2,3,8


In [3]:
!python ../scripts/create_summary.py

Traceback (most recent call last):
  File "C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\src\FIT_python\splits.py", line 31, in all_splits
    train_df, test_df = train_test_group_split(df)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\src\FIT_python\grouped_splits.py", line 31, in train_test_group_split
    train_idx, test_idx = next(sss.split(groups.reshape(-1,1), y_groups))
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\model_selection\_split.py", line 1927, in split
    for train, test in self._iter_indices(X, y, groups):
  File "c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\model_selection\_split.py", line 2342, in _iter_indices
    raise ValueError(
ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of 

In [4]:
!python ../scripts/plot_summary.py

Traceback (most recent call last):
  File "c:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\scripts\plot_summary.py", line 115, in <module>
    main()
  File "c:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\scripts\plot_summary.py", line 108, in main
    df = load_summary(DATA_PATH)
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\scripts\plot_summary.py", line 33, in load_summary
    df = pd.read_csv(path)
         ^^^^^^^^^^^^^^^^^
  File "c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\pandas\io\parsers\readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\pandas\io\parsers\readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Frede\an